In [15]:
import pandas as pd
import numpy as np
import time
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

RANDOM_STATE = 42

DATA_DIR = Path("../data/raw")
TRAIN_TRANSACTION_PATH = DATA_DIR / "train_transaction.csv"
TRAIN_IDENTITY_PATH = DATA_DIR / "train_identity.csv"

OUTPUT_METRICS_DIR = Path("../outputs/metrics")
MODEL_DIR = Path("../models/heavy")

TARGET_COL = "isFraud"

In [16]:
transaction = pd.read_csv(TRAIN_TRANSACTION_PATH)
identity = pd.read_csv(TRAIN_IDENTITY_PATH)

df = transaction.merge(
    identity,
    on="TransactionID",
    how="left"
)

print("Transaction shape:", transaction.shape)
print("Identity shape:", identity.shape)
print("Merged shape:", df.shape)
print("Merge satir sayisi dogru mu:", df.shape[0] == transaction.shape[0])

Transaction shape: (590540, 394)
Identity shape: (144233, 41)
Merged shape: (590540, 434)
Merge satir sayisi dogru mu: True


In [17]:
X = df.drop(columns=[TARGET_COL, "TransactionID"], errors="ignore")
y = df[TARGET_COL].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=RANDOM_STATE,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_valid.shape, y_valid.shape)
print("Test shape:", X_test.shape, y_test.shape)

print("Train fraud orani:", y_train.mean())
print("Validation fraud orani:", y_valid.mean())
print("Test fraud orani:", y_test.mean())

Train shape: (354324, 432) (354324,)
Validation shape: (118108, 432) (118108,)
Test shape: (118108, 432) (118108,)
Train fraud orani: 0.03499057359930459
Validation fraud orani: 0.0349933958749619
Test fraud orani: 0.03498492904798998


In [18]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "str", "category", "bool"]
).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Sayisal sutun sayisi:", len(numeric_features))
print("Kategorik sutun sayisi:", len(categorical_features))
print("Toplam ozellik sayisi:", len(numeric_features) + len(categorical_features))

Sayisal sutun sayisi: 401
Kategorik sutun sayisi: 31
Toplam ozellik sayisi: 432


In [19]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Normal sayisi:", negative_count)
print("Fraud sayisi:", positive_count)
print("Negatif / pozitif sinif orani:", scale_pos_weight)

Normal sayisi: 341926
Fraud sayisi: 12398
Negatif / pozitif sinif orani: 27.57912566542991


In [20]:
heavy_rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

start_time = time.perf_counter()

heavy_rf_model.fit(X_train, y_train)

training_time_sec = time.perf_counter() - start_time

print("Heavy Random Forest egitildi.")
print("Egitim suresi saniye:", training_time_sec)

Heavy Random Forest egitildi.
Egitim suresi saniye: 357.18306740000116


In [21]:
heavy_rf_val_prob = heavy_rf_model.predict_proba(X_valid)[:, 1]

print("Validation probability uretildi.")
print("Ilk 10 probability:")
print(heavy_rf_val_prob[:10])

Validation probability uretildi.
Ilk 10 probability:
[0.06511681 0.18849417 0.36610529 0.02829185 0.03102956 0.10742733
 0.22306514 0.06481961 0.09172739 0.36844019]


In [22]:
def evaluate_thresholds(model_name, y_true, y_prob):
    rows = []

    roc_auc = roc_auc_score(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)

    for threshold in np.arange(0.05, 0.55, 0.05):
        threshold = round(float(threshold), 2)
        y_pred = (y_prob >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        rows.append({
            "model": model_name,
            "threshold": threshold,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
            "routed_rate": y_pred.mean(),
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp
        })

    return pd.DataFrame(rows)

In [23]:
heavy_rf_threshold_results = evaluate_thresholds(
    model_name="Heavy Random Forest",
    y_true=y_valid,
    y_prob=heavy_rf_val_prob
)

heavy_rf_threshold_results

,model,threshold,precision,recall,f1,roc_auc,pr_auc,routed_rate,tn,fp,fn,tp
0,Heavy Random Forest,0.05,0.052408,0.989112,0.099542,0.935737,0.64282,0.660438,40060,73915,45,4088
1,Heavy Random Forest,0.10,0.081940,0.954754,0.150928,0.935737,0.64282,0.407737,69764,44211,187,3946
2,Heavy Random Forest,0.15,0.124678,0.913138,0.219399,0.935737,0.64282,0.256291,87479,26496,359,3774
3,Heavy Random Forest,0.20,0.179107,0.863295,0.296666,0.935737,0.64282,0.168668,97622,16353,565,3568
4,Heavy Random Forest,0.25,0.242540,0.812243,0.373540,0.935737,0.64282,0.117189,103491,10484,776,3357
5,Heavy Random Forest,0.30,0.313607,0.762884,0.444491,0.935737,0.64282,0.085125,107074,6901,980,3153
6,Heavy Random Forest,0.35,0.388238,0.706025,0.500987,0.935737,0.64282,0.063637,109377,4598,1215,2918
7,Heavy Random Forest,0.40,0.469487,0.655214,0.547015,0.935737,0.64282,0.048837,110915,3060,1425,2708
8,Heavy Random Forest,0.45,0.554922,0.612388,0.582241,0.935737,0.64282,0.038617,111945,2030,1602,2531
9,Heavy Random Forest,0.50,0.644878,0.568110,0.604065,0.935737,0.64282,0.030828,112682,1293,1785,2348


In [24]:
def measure_inference_time(model, X_data, sample_size=10000, repeat_count=5):
    sample = X_data.iloc[:min(sample_size, len(X_data))]

    times = []

    for _ in range(repeat_count):
        start = time.perf_counter()
        _ = model.predict_proba(sample)[:, 1]
        end = time.perf_counter()

        times.append(end - start)

    return {
        "sample_size": len(sample),
        "inference_mean_sec": np.mean(times),
        "inference_std_sec": np.std(times)
    }

In [25]:
heavy_rf_inference_time = measure_inference_time(
    model=heavy_rf_model,
    X_data=X_valid,
    sample_size=10000,
    repeat_count=5
)

heavy_rf_inference_time

{'sample_size': 10000,
 'inference_mean_sec': np.float64(0.5566153599997051),
 'inference_std_sec': np.float64(0.00776538095192448)}

In [26]:
selected_threshold = 0.15
selected_model_name = "Heavy Random Forest"

selected_test_prob = heavy_rf_model.predict_proba(X_test)[:, 1]
selected_test_pred = (selected_test_prob >= selected_threshold).astype(int)

test_confusion = confusion_matrix(y_test, selected_test_pred)
tn, fp, fn, tp = test_confusion.ravel()

test_metrics = {
    "model": selected_model_name,
    "threshold": selected_threshold,
    "precision": precision_score(y_test, selected_test_pred, zero_division=0),
    "recall": recall_score(y_test, selected_test_pred, zero_division=0),
    "f1": f1_score(y_test, selected_test_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, selected_test_prob),
    "pr_auc": average_precision_score(y_test, selected_test_prob),
    "routed_rate": selected_test_pred.mean(),
    "tn": tn,
    "fp": fp,
    "fn": fn,
    "tp": tp,
}

pd.DataFrame([test_metrics])

,model,threshold,precision,recall,f1,roc_auc,pr_auc,routed_rate,tn,fp,fn,tp
0,Heavy Random Forest,0.15,0.123495,0.908277,0.217427,0.933225,0.632287,0.257307,87339,26637,379,3753


In [27]:
fitted_preprocessor = heavy_rf_model.named_steps["preprocessor"]
fitted_rf = heavy_rf_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": fitted_rf.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(30)

,feature,importance
22,num__C13,0.026367
1,num__TransactionAmt,0.021684
23,num__C14,0.021673
0,num__TransactionDT,0.018071
2,num__card1,0.015370
3,num__card2,0.014020
38,num__D15,0.011639
10,num__C1,0.011224
6,num__addr1,0.011115
25,num__D2,0.010869


In [28]:
OUTPUT_METRICS_DIR.mkdir(parents=True, exist_ok=True)

heavy_rf_threshold_results["training_time_sec"] = training_time_sec
heavy_rf_threshold_results["inference_mean_sec"] = heavy_rf_inference_time["inference_mean_sec"]
heavy_rf_threshold_results["inference_std_sec"] = heavy_rf_inference_time["inference_std_sec"]

heavy_rf_results_path = OUTPUT_METRICS_DIR / "phase3_heavy_random_forest_threshold_results.csv"
heavy_rf_feature_importance_path = OUTPUT_METRICS_DIR / "phase3_heavy_random_forest_feature_importance.csv"
heavy_rf_test_metrics_path = OUTPUT_METRICS_DIR / "phase3_heavy_random_forest_test_metrics.csv"

heavy_rf_threshold_results.to_csv(heavy_rf_results_path, index=False)
feature_importance.to_csv(heavy_rf_feature_importance_path, index=False)
pd.DataFrame([test_metrics]).to_csv(heavy_rf_test_metrics_path, index=False)

print("Heavy Random Forest threshold sonuclari kaydedildi:")
print(heavy_rf_results_path)

print("Heavy Random Forest feature importance kaydedildi:")
print(heavy_rf_feature_importance_path)

print("Heavy Random Forest test metrikleri kaydedildi:")
print(heavy_rf_test_metrics_path)

Heavy Random Forest threshold sonuclari kaydedildi:
..\outputs\metrics\phase3_heavy_random_forest_threshold_results.csv
Heavy Random Forest feature importance kaydedildi:
..\outputs\metrics\phase3_heavy_random_forest_feature_importance.csv
Heavy Random Forest test metrikleri kaydedildi:
..\outputs\metrics\phase3_heavy_random_forest_test_metrics.csv


threshold = 0.15
recall = 0.913138
precision = 0.124678
f1 = 0.219399
routed_rate = 0.256291
Bu seçim fraud yakalama oranını %91 civarında tutuyor ve işlemlerin yaklaşık %25.6’sını riskli olarak yönlendiriyor.

In [29]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

selected_model_name = "Heavy Random Forest"
selected_threshold = 0.15
selected_model = heavy_rf_model

selected_model_metrics = heavy_rf_threshold_results[
    (heavy_rf_threshold_results["model"] == selected_model_name) &
    (heavy_rf_threshold_results["threshold"] == selected_threshold)
].iloc[0].to_dict()

model_bundle = {
    "pipeline": selected_model,
    "threshold": selected_threshold,
    "model_name": selected_model_name,
    "feature_columns": X_train.columns.tolist(),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "validation_metrics": selected_model_metrics,
    "test_metrics": test_metrics,
    "training_time_sec": training_time_sec,
    "inference_time": heavy_rf_inference_time,
    "feature_importance_top30": feature_importance.head(30).to_dict(orient="records"),
    "note": "Test verisi model seciminde kullanilmadi."
}

model_path = MODEL_DIR / "heavy_random_forest_model.joblib"

joblib.dump(model_bundle, model_path)

print("Heavy Random Forest modeli, preprocessing ve threshold birlikte kaydedildi:")
print(model_path)

Heavy Random Forest modeli, preprocessing ve threshold birlikte kaydedildi:
..\models\heavy\heavy_random_forest_model.joblib
